# Chapter 19: Euclidean Structures from a Projective Perspective

**Source orientation.** Perspectives on Projective Geometry, Chapter 19, sections 19.1-19.7; printed pp. 349-374; PDF pp. 371-396. The source span was used for terminology, coverage, and theorem order only. All prose, code, checks, and diagrams here are original.

**Chapter goal.** Rebuild familiar Euclidean data from projective primitives once the invisible circular points
\[
I=(1,i,0),\qquad J=(1,-i,0)
\]
are allowed to participate.

**Chapter question.** What Euclidean constructions become clearer when the invisible circular points are included?

The route is construction first. We use joins, meets, tangents, conics, and incidence checks, then ask what Euclidean statement the projective computation has recovered. The important habit is to treat `I` and `J` as ordinary projective points over the complex numbers even when no real drawing can show them.

In [ ]:
from pathlib import Path
import csv
import json
import math
import cmath
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from IPython.display import HTML, display

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives-on-Projective-Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, book_relative, display_artifact, ensure_artifact_root, image_stats, save_json, save_table

CHAPTER_SLUG = "chapter-19-euclidean-structures-from-a-projective-perspective"
ARTIFACT_ROOT = ensure_artifact_root(BOOK_ROOT / "artifacts" / CHAPTER_SLUG)
FIGURES = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"
TABLES = ARTIFACT_ROOT / "tables"
CHECKS_DIR = ARTIFACT_ROOT / "checks"

ARTIFACTS = []
CHECKS = {"chapter": 19, "source_span": "printed pp. 349-374 / PDF pp. 371-396"}
ARTIFACT_ROOT

## Computational Translation Guide

| Book idea | Computational object in this notebook |
| --- | --- |
| Circular points | Complex homogeneous points `I=(1,i,0)` and `J=(1,-i,0)` on the line at infinity. |
| Mirror image | Seven join/meet operations with `I`, `J`, the point, and the mirror line; compared to the ordinary reflection formula. |
| Angle bisector | A point `A` on the line at infinity whose Laguerre cross-ratio condition makes the two adjacent angles equal. |
| Circle center | The meet of the two complex tangents `Q I` and `Q J` to the circle conic matrix `Q`. |
| Conic focus | An intersection of tangents drawn from `I` and `J` to the conic. For real ellipses, two of the four intersections are real. |
| Triangle theorems | Concurrence and collinearity tests: medians via harmonic midpoint/Ceva, altitudes via orthogonality, angle bisectors via the same circular-point metric, Euler line via projective perspectivity. |
| Hybrid thinking | Replace the orthocenter in the nine-point circle theorem by an arbitrary point `H`; the circle relaxes to a conic that still contains the six midpoints and three cevian feet. |

**Library routing.** Matplotlib is used for durable planar construction diagrams; Plotly is used only where moving the boundary point changes the conics in a meaningful way; SymPy checks a small exact reflection identity; NetworkX records proof dependencies; NumPy and pandas carry the projective linear algebra and invariant tables.

In [ ]:
storyboard_path = CHECKS_DIR / "storyboard.json"
storyboard = json.loads(storyboard_path.read_text(encoding="utf-8"))
pd.DataFrame(storyboard["visual sequence"])[["order", "concept", "artifact_filename", "validation"]]

In [ ]:
I = np.array([1.0 + 0j, 1.0j, 0.0 + 0j])
J = np.array([1.0 + 0j, -1.0j, 0.0 + 0j])
LINE_INFINITY = np.array([0.0 + 0j, 0.0 + 0j, 1.0 + 0j])
TOL = 1e-8


def hxy(x, y):
    return np.array([complex(x), complex(y), 1.0 + 0j])


def join(p, q):
    return np.cross(np.asarray(p, dtype=complex), np.asarray(q, dtype=complex))


def meet(l, m):
    return np.cross(np.asarray(l, dtype=complex), np.asarray(m, dtype=complex))


def bracket(a, b, c):
    return np.linalg.det(np.column_stack([a, b, c]))


def line_from_point_direction(point, direction):
    p = hxy(point[0], point[1])
    q = hxy(point[0] + direction[0], point[1] + direction[1])
    return join(p, q)


def normalize_by_largest(v):
    arr = np.asarray(v, dtype=complex)
    idx = int(np.argmax(np.abs(arr)))
    if abs(arr[idx]) > 1e-14:
        arr = arr * np.exp(-1j * np.angle(arr[idx]))
        arr = arr / arr[idx].real
    return arr


def affine_real(point, tol=1e-8):
    arr = np.asarray(point, dtype=complex)
    if abs(arr[2]) < 1e-12:
        raise ValueError("point at infinity")
    arr = arr / arr[2]
    if np.max(np.abs(arr.imag)) > tol:
        raise ValueError(f"point has non-real affine residue {np.max(np.abs(arr.imag))}")
    return arr.real[:2]


def real_direction_from_infinite(point, tol=1e-8):
    arr = normalize_by_largest(point)
    if np.max(np.abs(arr.imag)) > tol:
        raise ValueError(f"direction has non-real residue {np.max(np.abs(arr.imag))}")
    d = arr.real[:2]
    return d / np.linalg.norm(d)


def reflect_point_across_line(point, line):
    a, b, c = np.asarray(line, dtype=complex).real
    x, y = point
    signed = (a * x + b * y + c) / (a * a + b * b)
    return np.array([x - 2 * a * signed, y - 2 * b * signed])


def draw_line(ax, line, xlim, ylim, **kwargs):
    a, b, c = np.asarray(line, dtype=complex).real
    if abs(b) > abs(a):
        xs = np.array(xlim)
        ys = (-a * xs - c) / b
        ax.plot(xs, ys, **kwargs)
    else:
        ys = np.array(ylim)
        xs = (-b * ys - c) / a
        ax.plot(xs, ys, **kwargs)


def draw_segment(ax, p, q, **kwargs):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    ax.plot([p[0], q[0]], [p[1], q[1]], **kwargs)


def label_point(ax, xy, label, dx=0.04, dy=0.04, **kwargs):
    ax.scatter([xy[0]], [xy[1]], s=42, zorder=5, **{k: v for k, v in kwargs.items() if k in {"color", "c"}})
    ax.text(xy[0] + dx, xy[1] + dy, label, fontsize=10, weight="bold")


def save_fig(fig, filename, dpi=170):
    path = FIGURES / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    ARTIFACTS.append(path)
    return path


def fit_conic(points):
    rows = []
    for x, y in points:
        rows.append([x * x, y * y, 1.0, 2.0 * x * y, 2.0 * x, 2.0 * y])
    _, _, vh = np.linalg.svd(np.asarray(rows, dtype=float))
    a, b, c, d, e, f = vh[-1]
    return np.array([[a, d, e], [d, b, f], [e, f, c]], dtype=float)


def eval_conic(Q, p):
    v = np.array([p[0], p[1], 1.0], dtype=float)
    return float(v @ Q @ v)


def real_line(p, q):
    return np.cross(np.r_[p, 1.0], np.r_[q, 1.0])


def real_meet(l, m):
    p = np.cross(l, m)
    return p[:2] / p[2]


def cross_ratio(a, b, c, d):
    return ((a - c) * (b - d)) / ((a - d) * (b - c))


def unit(v):
    v = np.asarray(v, dtype=float)
    return v / np.linalg.norm(v)


def angle_between(u, v):
    dot = float(np.clip(np.dot(unit(u), unit(v)), -1.0, 1.0))
    return math.acos(dot)


def conic_grid(Q, samples=260):
    xs = np.linspace(-2.2, 2.6, samples)
    ys = np.linspace(-1.7, 2.1, samples)
    xx, yy = np.meshgrid(xs, ys)
    zz = Q[0, 0] * xx * xx + Q[1, 1] * yy * yy + Q[2, 2] + 2 * Q[0, 1] * xx * yy + 2 * Q[0, 2] * xx + 2 * Q[1, 2] * yy
    return xx, yy, zz

## 1. Mirror Image From Circular Points

The mirror construction is projective in form: connect the point to `I` and `J`, meet those two lines with the mirror, then cross-connect with `J` and `I`. The result should be the ordinary Euclidean reflection even though the intermediate circular-point lines are complex.

Inspection target: the computed point `p'` should sit on the perpendicular through `p`, and the midpoint of `p` and `p'` should lie on the mirror line.

In [ ]:
p = hxy(1.60, 1.25)
mirror_angle = 0.55
mirror_direction = np.array([math.cos(mirror_angle), math.sin(mirror_angle)])
mirror_line = line_from_point_direction(np.array([-0.20, -0.70]), mirror_direction)

lpI = join(p, I)
lpJ = join(p, J)
a = meet(lpI, mirror_line)
b = meet(lpJ, mirror_line)
p_reflected_projective = meet(join(b, I), join(a, J))
p_reflected = affine_real(p_reflected_projective)
p_reflected_classical = reflect_point_across_line(np.array([p[0].real, p[1].real]), mirror_line)
mid = 0.5 * (np.array([p[0].real, p[1].real]) + p_reflected)

mirror_checks = {
    "projective_vs_classical_reflection": float(np.linalg.norm(p_reflected - p_reflected_classical)),
    "midpoint_on_mirror_line": float(abs(np.dot(mirror_line.real, np.r_[mid, 1.0]))),
    "reflection_segment_perpendicular_to_mirror": float(abs(np.dot(p_reflected - np.array([p[0].real, p[1].real]), mirror_direction))),
    "reflected_point_imaginary_residual": float(np.max(np.abs((p_reflected_projective / p_reflected_projective[2]).imag))),
}
CHECKS["mirror"] = mirror_checks

xlim, ylim = (-0.8, 2.8), (-1.4, 1.8)
fig, ax = plt.subplots(figsize=(7.6, 5.2))
draw_line(ax, mirror_line, xlim, ylim, color="#2b6cb0", linewidth=2.4, label="mirror line l")
draw_segment(ax, [p[0].real, p[1].real], p_reflected, color="#805ad5", linewidth=2.0, linestyle="--", label="join(p,p')")
label_point(ax, [p[0].real, p[1].real], "p", color="#d9480f")
label_point(ax, p_reflected, "p'", color="#2f9e44")
label_point(ax, mid, "m", color="#2b6cb0")
ax.scatter([a[0].real / a[2].real, b[0].real / b[2].real], [a[1].real / a[2].real, b[1].real / b[2].real], color="#111827", s=34, zorder=4)
ax.text(a[0].real / a[2].real + 0.04, a[1].real / a[2].real + 0.04, "a", fontsize=9)
ax.text(b[0].real / b[2].real + 0.04, b[1].real / b[2].real + 0.04, "b", fontsize=9)
ax.annotate("complex joins to I and J\nmeet l at a and b", xy=(0.05, 1.45), xytext=(0.55, 1.45), arrowprops={"arrowstyle": "->", "color": "#4a5568"}, fontsize=9)
ax.text(1.68, -1.23, "checks:\nmidpoint on l = %.1e\nperpendicular = %.1e" % (mirror_checks["midpoint_on_mirror_line"], mirror_checks["reflection_segment_perpendicular_to_mirror"]), fontsize=9, bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#cbd5e1"})
ax.set_title("Mirror image from joins and meets with circular points")
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)
ax.legend(loc="lower left")
mirror_artifact = save_fig(fig, "mirror-image-circular-points.png")
mirror_artifact

In [ ]:
display_artifact(mirror_artifact, width=760)

A small exact check clarifies why this is not just a lucky numeric example. For the horizontal mirror `y=0`, the same seven projective operations send `(x,y,1)` to `(x,-y,1)`.

In [ ]:
xs, ys = sp.symbols("x y", real=True)
Is = sp.Matrix([1, sp.I, 0])
Js = sp.Matrix([1, -sp.I, 0])
ps = sp.Matrix([xs, ys, 1])
ls = sp.Matrix([0, 1, 0])

def sx(u, v):
    return sp.Matrix([u[1] * v[2] - u[2] * v[1], u[2] * v[0] - u[0] * v[2], u[0] * v[1] - u[1] * v[0]])

asym = sx(sx(ps, Is), ls)
bsym = sx(sx(ps, Js), ls)
pprime_sym = sx(sx(bsym, Is), sx(asym, Js))
pprime_sym = sp.simplify(pprime_sym / pprime_sym[2])
exact_reflection_ok = [sp.simplify(v) for v in (pprime_sym - sp.Matrix([xs, -ys, 1]))] == [0, 0, 0]
CHECKS["mirror"]["symbolic_horizontal_reflection_ok"] = bool(exact_reflection_ok)
pprime_sym, exact_reflection_ok

## 2. Angle Bisectors via Laguerre's Formula

The angle between real lines is encoded at infinity by their intersections `L` and `M` with `l_infinity`, measured relative to `I` and `J`. An angle bisector has an infinite point `A` satisfying the equal-angle cross-ratio condition. The bracket formula produces two choices, corresponding to the two perpendicular bisectors of the angle pair.

Inspection target: the bracket-derived rays should lie exactly on the classical internal and external angle bisectors.

In [ ]:
theta_l = 0.18
theta_m = 1.43
O = np.array([-0.20, -0.15])
dir_l = np.array([math.cos(theta_l), math.sin(theta_l)])
dir_m = np.array([math.cos(theta_m), math.sin(theta_m)])
L = np.array([dir_l[0] + 0j, dir_l[1] + 0j, 0.0 + 0j])
M = np.array([dir_m[0] + 0j, dir_m[1] + 0j, 0.0 + 0j])
P_generic = hxy(0.37, -0.91)

lam = cmath.sqrt(bracket(P_generic, L, J) * bracket(P_generic, M, J))
mu = cmath.sqrt(bracket(P_generic, L, I) * bracket(P_generic, M, I))
A_plus = lam * I + mu * J
A_minus = lam * I - mu * J
bisector_dirs = [real_direction_from_infinite(A_plus), real_direction_from_infinite(A_minus)]
classical_dirs = [unit(dir_l + dir_m), unit(dir_l - dir_m)]

mismatches = []
for d in bisector_dirs:
    mismatches.append(min(1 - abs(float(np.dot(d, c))) for c in classical_dirs))
angle_checks = {
    "max_direction_mismatch": float(max(mismatches)),
    "bisectors_perpendicular_dot": float(abs(np.dot(bisector_dirs[0], bisector_dirs[1]))),
    "equal_angle_residual_internal": float(abs(angle_between(dir_l, bisector_dirs[1]) - angle_between(bisector_dirs[1], dir_m))),
}
CHECKS["angle_bisectors"] = angle_checks

fig, ax = plt.subplots(figsize=(7.2, 5.6))
xlim, ylim = (-2.2, 2.2), (-1.8, 2.1)
for direction, label, color in [(dir_l, "l", "#2b6cb0"), (dir_m, "m", "#2b6cb0")]:
    pts = np.array([O - 2.3 * direction, O + 2.3 * direction])
    ax.plot(pts[:, 0], pts[:, 1], color=color, linewidth=2.0)
    ax.text(*(O + 1.82 * direction), label, fontsize=11, weight="bold", color=color)
for direction, label, color in [(bisector_dirs[0], "a-", "#d9480f"), (bisector_dirs[1], "a+", "#2f9e44")]:
    pts = np.array([O - 2.3 * direction, O + 2.3 * direction])
    ax.plot(pts[:, 0], pts[:, 1], color=color, linewidth=2.3, linestyle="--")
    ax.text(*(O + 1.45 * direction), label, fontsize=10, weight="bold", color=color)
label_point(ax, O, "O", color="#111827")
ax.text(-2.05, 1.55, "A = sqrt([P,L,J][P,M,J]) I +/- sqrt([P,L,I][P,M,I]) J\nmax direction mismatch = %.1e\nbisector dot product = %.1e" % (angle_checks["max_direction_mismatch"], angle_checks["bisectors_perpendicular_dot"]), fontsize=8.8, bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#cbd5e1"})
ax.set_title("Angle bisectors from circular-point bracket data")
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.25)
angle_artifact = save_fig(fig, "angle-bisectors-laguerre-formula.png")
angle_artifact

In [ ]:
display_artifact(angle_artifact, width=760)

## 3. Center of a Circle From Invisible Tangents

For a circle with conic matrix `Q`, the tangent at a point `X` on the conic is the line `Q X`. Since every circle contains `I` and `J`, the lines `Q I` and `Q J` are legitimate tangents, even though they are complex. Their meet is real and equals the Euclidean center.

Inspection target: no real tangent lines are drawn at `I` or `J`; the diagram shows the real consequence of their complex meet.

In [ ]:
def circle_matrix(cx, cy, r):
    return np.array([[1.0, 0.0, -cx], [0.0, 1.0, -cy], [-cx, -cy, cx * cx + cy * cy - r * r]], dtype=complex)

circle_center = np.array([0.55, -0.22])
radius = 1.18
Q_circle = circle_matrix(circle_center[0], circle_center[1], radius)
tangent_I = Q_circle @ I
tangent_J = Q_circle @ J
center_projective = meet(tangent_I, tangent_J)
center_recovered = affine_real(center_projective)

circle_checks = {
    "center_residual": float(np.linalg.norm(center_recovered - circle_center)),
    "center_imaginary_residual": float(np.max(np.abs((center_projective / center_projective[2]).imag))),
    "QI_dot_I": float(abs(np.dot(tangent_I, I))),
    "QJ_dot_J": float(abs(np.dot(tangent_J, J))),
}
CHECKS["circle_center"] = circle_checks

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.8), gridspec_kw={"width_ratios": [1.05, 1.0]})
ax = axes[0]
t = np.linspace(0, 2 * math.pi, 400)
ax.plot(circle_center[0] + radius * np.cos(t), circle_center[1] + radius * np.sin(t), color="#2b6cb0", linewidth=2.4)
label_point(ax, circle_center, "given center", color="#94a3b8", dx=-0.62, dy=-0.16)
label_point(ax, center_recovered, "meet(QI,QJ)", color="#d9480f", dx=0.08, dy=0.08)
ax.set_title("Real circle data")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.1, 2.15)
ax.set_ylim(-1.65, 1.35)
ax.grid(True, alpha=0.25)

ax = axes[1]
ax.axis("off")
lines = [
    "Complex tangent coefficients:",
    "QI = [%.2f%+.2fi, %.2f%+.2fi, %.2f%+.2fi]" % (tangent_I[0].real, tangent_I[0].imag, tangent_I[1].real, tangent_I[1].imag, tangent_I[2].real, tangent_I[2].imag),
    "QJ = [%.2f%+.2fi, %.2f%+.2fi, %.2f%+.2fi]" % (tangent_J[0].real, tangent_J[0].imag, tangent_J[1].real, tangent_J[1].imag, tangent_J[2].real, tangent_J[2].imag),
    "",
    "Their meet has real affine coordinates:",
    "(%.6f, %.6f)" % tuple(center_recovered),
    "",
    "center residual = %.1e" % circle_checks["center_residual"],
]
ax.text(0.02, 0.95, "\n".join(lines), va="top", ha="left", family="monospace", fontsize=9.2, bbox={"boxstyle": "round,pad=0.55", "facecolor": "#f8fafc", "edgecolor": "#cbd5e1"})
fig.suptitle("Circle center as the meet of invisible tangents", y=1.02)
circle_artifact = save_fig(fig, "circle-center-invisible-tangents.png")
circle_artifact

In [ ]:
display_artifact(circle_artifact, width=840)

## 4. Foci of a Conic From Tangents Through `I` and `J`

For an ellipse `x^2/a^2 + y^2/b^2 = 1`, the tangents from `I` and `J` are complex. Intersecting them in the four possible pairings gives four projective foci; exactly two are real for this ellipse.

Inspection target: the real intersections match the classical foci, and the sampled ray segments satisfy the equal-angle reflection law at the ellipse.

In [ ]:
a_axis = 2.35
b_axis = 1.18
focus_c = math.sqrt(a_axis * a_axis - b_axis * b_axis)
I_tangents = [np.array([-1j, 1.0 + 0j, 1j * focus_c]), np.array([-1j, 1.0 + 0j, -1j * focus_c])]
J_tangents = [np.array([1j, 1.0 + 0j, 1j * focus_c]), np.array([1j, 1.0 + 0j, -1j * focus_c])]
focus_candidates = []
for i_idx, li in enumerate(I_tangents):
    for j_idx, lj in enumerate(J_tangents):
        point = meet(li, lj)
        normalized = point / point[2]
        focus_candidates.append({"I_tangent": i_idx + 1, "J_tangent": j_idx + 1, "point": normalized})
real_foci = [affine_real(item["point"]) for item in focus_candidates if np.max(np.abs(item["point"].imag)) < 1e-8]
real_foci = sorted(real_foci, key=lambda xy: xy[0])
F1, F2 = real_foci[0], real_foci[1]

ray_ts = np.linspace(0.42, 2.52, 6)
reflection_residuals = []
for theta in ray_ts:
    P = np.array([a_axis * math.cos(theta), b_axis * math.sin(theta)])
    normal = unit(np.array([P[0] / (a_axis * a_axis), P[1] / (b_axis * b_axis)]))
    u1 = unit(F1 - P)
    u2 = unit(F2 - P)
    bisector = unit(u1 + u2)
    reflection_residuals.append(abs(np.cross(bisector, normal)))

focus_checks = {
    "focus_coordinate_residual": float(max(np.linalg.norm(F1 - np.array([-focus_c, 0.0])), np.linalg.norm(F2 - np.array([focus_c, 0.0])))),
    "max_reflection_law_residual": float(max(reflection_residuals)),
    "real_focus_count": len(real_foci),
    "complex_focus_count": 4 - len(real_foci),
}
CHECKS["conic_foci"] = focus_checks

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.0), gridspec_kw={"width_ratios": [1.25, 0.95]})
ax = axes[0]
theta = np.linspace(0, 2 * math.pi, 500)
ellipse = np.column_stack([a_axis * np.cos(theta), b_axis * np.sin(theta)])
ax.plot(ellipse[:, 0], ellipse[:, 1], color="#2b6cb0", linewidth=2.4, label="conic C")
label_point(ax, F1, "F1", color="#d9480f")
label_point(ax, F2, "F2", color="#d9480f")
for theta in ray_ts:
    P = np.array([a_axis * math.cos(theta), b_axis * math.sin(theta)])
    draw_segment(ax, F1, P, color="#64748b", linewidth=1.1, alpha=0.8)
    draw_segment(ax, P, F2, color="#2f9e44", linewidth=1.1, alpha=0.85)
    ax.scatter([P[0]], [P[1]], s=18, color="#111827", zorder=4)
ax.set_title("Real foci and reflected rays")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-2.7, 2.7)
ax.set_ylim(-1.55, 1.65)
ax.grid(True, alpha=0.25)
ax.legend(loc="lower center")

ax = axes[1]
ax.axis("off")
rows = ["Tangents from I and J to C:"]
for idx, line in enumerate(I_tangents, start=1):
    rows.append("i%d: %.0fi x + y %+ .3fi = 0" % (idx, line[0].imag, line[2].imag))
for idx, line in enumerate(J_tangents, start=1):
    rows.append("j%d: %.0fi x + y %+ .3fi = 0" % (idx, line[0].imag, line[2].imag))
rows.extend(["", "Real meets:", "F1 = (%.6f, %.6f)" % tuple(F1), "F2 = (%.6f, %.6f)" % tuple(F2), "", "focus residual = %.1e" % focus_checks["focus_coordinate_residual"], "reflection residual = %.1e" % focus_checks["max_reflection_law_residual"]])
ax.text(0.02, 0.95, "\n".join(rows), va="top", ha="left", family="monospace", fontsize=9.1, bbox={"boxstyle": "round,pad=0.55", "facecolor": "#f8fafc", "edgecolor": "#cbd5e1"})
fig.suptitle("Conic foci from circular-point tangent intersections", y=1.02)
foci_artifact = save_fig(fig, "conic-foci-from-circular-tangents.png")
foci_artifact

In [ ]:
display_artifact(foci_artifact, width=860)

## 5. Constructing Conics From Foci and One Boundary Point

The projective construction says: if `F1` and `F2` are the foci, the four lines through `F1,F2` and `I,J` are fixed tangents in the dual problem. In the real Euclidean slice, the two resulting conics through a boundary point are the confocal ellipse and hyperbola selected by constant distance sum and constant distance difference.

Inspection target: move the boundary point in the HTML artifact. The foci stay fixed; the ellipse and hyperbola both pass through the selected point.

In [ ]:
def confocal_curves(point, c=focus_c, samples=360):
    point = np.asarray(point, dtype=float)
    left = np.array([-c, 0.0])
    right = np.array([c, 0.0])
    d_left = np.linalg.norm(point - left)
    d_right = np.linalg.norm(point - right)
    ae = 0.5 * (d_left + d_right)
    be = math.sqrt(max(ae * ae - c * c, 1e-12))
    ah = 0.5 * abs(d_left - d_right)
    bh = math.sqrt(max(c * c - ah * ah, 1e-12))
    t = np.linspace(0, 2 * math.pi, samples)
    ellipse = np.column_stack([ae * np.cos(t), be * np.sin(t)])
    u = np.linspace(-1.75, 1.75, samples // 2)
    right_branch = np.column_stack([ah * np.cosh(u), bh * np.sinh(u)])
    left_branch = np.column_stack([-ah * np.cosh(u), bh * np.sinh(u)])
    hyperbola = np.vstack([right_branch, [[np.nan, np.nan]], left_branch])
    return ellipse, hyperbola, {"ellipse_sum": d_left + d_right, "hyperbola_difference": abs(d_left - d_right), "ae": ae, "be": be, "ah": ah, "bh": bh}

boundary_points = [np.array([1.60 + 0.22 * math.cos(t), y]) for t, y in zip(np.linspace(0, 2 * math.pi, 8, endpoint=False), np.linspace(-1.05, 1.15, 8))]
initial_point = boundary_points[4]
ellipse0, hyperbola0, params0 = confocal_curves(initial_point)

fig = go.Figure(
    data=[
        go.Scatter(x=ellipse0[:, 0], y=ellipse0[:, 1], mode="lines", name="ellipse solution", line={"color": "#2b6cb0", "width": 3}),
        go.Scatter(x=hyperbola0[:, 0], y=hyperbola0[:, 1], mode="lines", name="hyperbola solution", line={"color": "#d9480f", "width": 3}),
        go.Scatter(x=[-focus_c, focus_c], y=[0, 0], mode="markers+text", text=["F1", "F2"], textposition="bottom center", name="foci", marker={"color": "#111827", "size": 10}),
        go.Scatter(x=[initial_point[0]], y=[initial_point[1]], mode="markers+text", text=["P"], textposition="top center", name="boundary point", marker={"color": "#2f9e44", "size": 11}),
    ]
)
frames = []
for idx, point in enumerate(boundary_points):
    ellipse, hyperbola, params = confocal_curves(point)
    frames.append(go.Frame(name=str(idx), data=[
        go.Scatter(x=ellipse[:, 0], y=ellipse[:, 1], mode="lines", line={"color": "#2b6cb0", "width": 3}),
        go.Scatter(x=hyperbola[:, 0], y=hyperbola[:, 1], mode="lines", line={"color": "#d9480f", "width": 3}),
        go.Scatter(x=[-focus_c, focus_c], y=[0, 0], mode="markers+text", text=["F1", "F2"], textposition="bottom center", marker={"color": "#111827", "size": 10}),
        go.Scatter(x=[point[0]], y=[point[1]], mode="markers+text", text=["P"], textposition="top center", marker={"color": "#2f9e44", "size": 11}),
    ], layout=go.Layout(title_text="Confocal conics through P%d: sum %.3f, difference %.3f" % (idx + 1, params["ellipse_sum"], params["hyperbola_difference"]))))
fig.frames = frames
fig.update_layout(
    title="Two conics with prescribed foci through a moving point",
    width=820,
    height=560,
    xaxis={"range": [-3.2, 3.2], "scaleanchor": "y", "zeroline": False},
    yaxis={"range": [-2.25, 2.25], "zeroline": False},
    margin={"l": 40, "r": 20, "t": 70, "b": 40},
    updatemenus=[{"type": "buttons", "showactive": False, "x": 0.02, "y": 1.08, "buttons": [{"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": 750, "redraw": True}, "fromcurrent": True}]}, {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]}]}],
    sliders=[{"steps": [{"label": f"P{idx + 1}", "method": "animate", "args": [[str(idx)], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}]} for idx in range(len(boundary_points))]}],
)
html_artifact = HTML_DIR / "conics-through-point-from-foci.html"
fig.write_html(html_artifact, include_plotlyjs="inline", full_html=True)
ARTIFACTS.append(html_artifact)

xp, yp = initial_point
ellipse_value = xp * xp / (params0["ae"] ** 2) + yp * yp / (params0["be"] ** 2) - 1.0
hyperbola_value = xp * xp / (params0["ah"] ** 2) - yp * yp / (params0["bh"] ** 2) - 1.0
conics_from_foci_checks = {
    "initial_point_ellipse_equation_residual": float(abs(ellipse_value)),
    "initial_point_hyperbola_equation_residual": float(abs(hyperbola_value)),
    "ellipse_distance_sum": float(params0["ellipse_sum"]),
    "hyperbola_distance_difference": float(params0["hyperbola_difference"]),
    "html_file_size": int(html_artifact.stat().st_size),
}
CHECKS["conics_from_foci"] = conics_from_foci_checks
book_relative(html_artifact), conics_from_foci_checks

In [ ]:
display_artifact(html_artifact, width="100%", height=560)

## 6. Triangle Theorems as Projective Incidence

The chapter uses elementary triangle theorems as a translation test. Medians need only harmonic midpoint data. Altitudes and angle bisectors additionally need the Euclidean metric data carried by `I` and `J`. The Euler line then appears as a collinearity statement after the relevant centers have been built.

Inspection target: each panel names the Euclidean theorem and the projective data being checked: concurrence or collinearity.

In [ ]:
A = np.array([-1.45, -0.78])
B = np.array([2.05, -0.45])
C = np.array([0.18, 1.72])
X = 0.5 * (B + C)
Y = 0.5 * (C + A)
Z = 0.5 * (A + B)
G = (A + B + C) / 3.0

def altitude_line(vertex, side_p, side_q):
    side = side_q - side_p
    normal_direction = np.array([-side[1], side[0]])
    return real_line(vertex, vertex + normal_direction)

def perpendicular_bisector(p, q):
    mid = 0.5 * (p + q)
    side = q - p
    perp = np.array([-side[1], side[0]])
    return real_line(mid, mid + perp)

alt_A = altitude_line(A, B, C)
alt_B = altitude_line(B, C, A)
alt_C = altitude_line(C, A, B)
H = real_meet(alt_A, alt_B)
perp_AB = perpendicular_bisector(A, B)
perp_AC = perpendicular_bisector(A, C)
O_circum = real_meet(perp_AB, perp_AC)
side_a = np.linalg.norm(B - C)
side_b = np.linalg.norm(C - A)
side_c = np.linalg.norm(A - B)
In = (side_a * A + side_b * B + side_c * C) / (side_a + side_b + side_c)

def point_line_residual(point, line):
    return float(abs(np.dot(line, np.r_[point, 1.0])))

median_residual = np.linalg.norm(real_meet(real_line(A, X), real_line(B, Y)) - G)
altitude_residual = point_line_residual(H, alt_C)
incenter_residual = max(abs(angle_between(B - A, In - A) - angle_between(In - A, C - A)), abs(angle_between(C - B, In - B) - angle_between(In - B, A - B)))
euler_line = real_line(O_circum, H)
euler_residual = point_line_residual(G, euler_line)
triangle_checks = {
    "median_concurrence_residual": float(median_residual),
    "altitude_concurrence_residual": float(altitude_residual),
    "angle_bisector_equal_angle_residual": float(incenter_residual),
    "euler_line_collinearity_residual": float(euler_residual),
}
CHECKS["triangle_theorems"] = triangle_checks

fig, axes = plt.subplots(2, 2, figsize=(10.5, 8.6))
panels = [
    (axes[0, 0], "Medians: harmonic midpoint -> Ceva", [(A, X, "#2b6cb0"), (B, Y, "#2b6cb0"), (C, Z, "#2b6cb0")], [(G, "G")]),
    (axes[0, 1], "Altitudes: orthogonality uses I,J", [(A, H, "#d9480f"), (B, H, "#d9480f"), (C, H, "#d9480f")], [(H, "H")]),
    (axes[1, 0], "Angle bisectors: Laguerre data at infinity", [(A, In, "#2f9e44"), (B, In, "#2f9e44"), (C, In, "#2f9e44")], [(In, "I_c")]),
    (axes[1, 1], "Euler line: O, G, H are collinear", [(O_circum, H, "#805ad5"), (A, X, "#94a3b8"), (B, Y, "#94a3b8")], [(O_circum, "O"), (G, "G"), (H, "H")]),
]

for ax, title, lines, points in panels:
    triangle = np.vstack([A, B, C, A])
    ax.plot(triangle[:, 0], triangle[:, 1], color="#111827", linewidth=1.5)
    for label, point in [("A", A), ("B", B), ("C", C)]:
        label_point(ax, point, label, color="#111827")
    for p0, p1, color in lines:
        draw_segment(ax, p0, p1, color=color, linewidth=1.9, alpha=0.95)
    for point, label in points:
        label_point(ax, point, label, color="#d9480f" if label in {"H", "O"} else "#2b6cb0")
    ax.set_title(title, fontsize=11)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.85, 2.35)
    ax.set_ylim(-1.1, 2.05)
    ax.grid(True, alpha=0.22)

axes[1, 1].text(-1.75, -0.95, "Euler residual = %.1e" % euler_residual, fontsize=9, bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "#cbd5e1"})
fig.suptitle("Triangle theorems translated into projective checks", y=1.01)
triangle_artifact = save_fig(fig, "triangle-theorems-projective-dashboard.png")
triangle_artifact

In [ ]:
display_artifact(triangle_artifact, width=860)

## 7. Applied Lab: The Nine-Point Circle Relaxes to a Conic

The hybrid method is not to force everything into one language. Start with the Euclidean nine-point circle theorem, then remove the metric hypothesis that `H` is the orthocenter. For an arbitrary point `H`, the six midpoints of `AB, BC, CA, AH, BH, CH` still determine a conic. The three cevian feet `AH cap BC`, `BH cap CA`, and `CH cap AB` also lie on that conic.

Lab prompt: move `H` in the table below. The conic changes, but the three foot-point residuals should remain near zero. When `H` happens to be the orthocenter, this conic becomes the usual nine-point circle.

In [ ]:
def ninepoint_configuration(H_point):
    H_point = np.asarray(H_point, dtype=float)
    X = 0.5 * (B + C)
    Y = 0.5 * (C + A)
    Z = 0.5 * (A + B)
    Pm = 0.5 * (A + H_point)
    Qm = 0.5 * (B + H_point)
    Rm = 0.5 * (C + H_point)
    D = real_meet(real_line(A, H_point), real_line(B, C))
    E = real_meet(real_line(B, H_point), real_line(C, A))
    F = real_meet(real_line(C, H_point), real_line(A, B))
    midpoint_points = [X, Y, Z, Pm, Qm, Rm]
    foot_points = [D, E, F]
    Q = fit_conic(midpoint_points)
    return {"H": H_point, "midpoints": midpoint_points, "feet": foot_points, "Q": Q}

H_lab = np.array([0.55, 0.25])
config = ninepoint_configuration(H_lab)
Q_lab = config["Q"]
foot_residuals = [abs(eval_conic(Q_lab, p)) for p in config["feet"]]
midpoint_residuals = [abs(eval_conic(Q_lab, p)) for p in config["midpoints"]]

rows = []
for idx, H_candidate in enumerate([np.array([0.55, 0.25]), np.array([0.15, 0.92]), np.array([1.05, 0.48]), np.array([-0.10, 0.12])], start=1):
    cfg = ninepoint_configuration(H_candidate)
    foot_vals = [abs(eval_conic(cfg["Q"], p)) for p in cfg["feet"]]
    midpoint_vals = [abs(eval_conic(cfg["Q"], p)) for p in cfg["midpoints"]]
    rows.append({
        "case": idx,
        "H_x": float(H_candidate[0]),
        "H_y": float(H_candidate[1]),
        "max_midpoint_residual": float(max(midpoint_vals)),
        "max_foot_residual": float(max(foot_vals)),
    })
residual_table = save_table(rows, ARTIFACT_ROOT, "tables", "hybrid-ninepoint-residuals.csv")
ARTIFACTS.append(residual_table)

xx, yy, zz = conic_grid(Q_lab)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), gridspec_kw={"width_ratios": [1.25, 0.95]})
ax = axes[0]
triangle = np.vstack([A, B, C, A])
ax.plot(triangle[:, 0], triangle[:, 1], color="#111827", linewidth=1.5)
for p0, p1 in [(A, H_lab), (B, H_lab), (C, H_lab)]:
    draw_segment(ax, p0, p1, color="#64748b", linewidth=1.1, linestyle="--")
ax.contour(xx, yy, zz, levels=[0], colors=["#2b6cb0"], linewidths=2.4)
for label, point in [("A", A), ("B", B), ("C", C), ("H", H_lab)]:
    label_point(ax, point, label, color="#111827" if label != "H" else "#d9480f")
for label, point in zip(["X", "Y", "Z", "P", "Q", "R"], config["midpoints"]):
    label_point(ax, point, label, color="#2b6cb0", dx=0.035, dy=0.035)
for label, point in zip(["D", "E", "F"], config["feet"]):
    label_point(ax, point, label, color="#2f9e44", dx=0.035, dy=-0.085)
ax.set_title("Arbitrary H: six midpoint points plus three cevian feet")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.85, 2.35)
ax.set_ylim(-1.1, 2.05)
ax.grid(True, alpha=0.22)

ax = axes[1]
ax.axis("off")
df_resid = pd.DataFrame(rows)
ax.text(0.02, 0.96, "Residual table over H choices", fontsize=11, weight="bold", va="top")
ax.text(0.02, 0.86, df_resid.to_string(index=False, float_format=lambda x: f"{x:.2e}"), family="monospace", fontsize=8.8, va="top", bbox={"boxstyle": "round,pad=0.5", "facecolor": "#f8fafc", "edgecolor": "#cbd5e1"})
ax.text(0.02, 0.20, "The conic is projective data.\nIt becomes a circle only after adding\nthe Euclidean condition that H is the orthocenter.", fontsize=9.4, va="top")
fig.suptitle("Hybrid lab: the nine-point circle as a relaxed conic", y=1.02)
hybrid_artifact = save_fig(fig, "hybrid-ninepoint-conic-lab.png")

hybrid_checks = {
    "max_midpoint_residual": float(max(midpoint_residuals)),
    "max_foot_residual": float(max(foot_residuals)),
    "table_max_foot_residual": float(max(row["max_foot_residual"] for row in rows)),
}
CHECKS["hybrid_ninepoint_conic"] = hybrid_checks
hybrid_artifact, df_resid

In [ ]:
display_artifact(hybrid_artifact, width=860)
display_artifact(residual_table)

## 8. Proof Dependency Scaffold

The chapter is a sampler rather than one long proof. The dependency graph below keeps the proof moves visible: circular points feed metric notions; mirror construction feeds foci; bracket and quadrilateral-set conditions feed angle and triangle facts; Pascal, Brianchon, Hesse transfer, and Desargues explain why incidence-only arguments can recover Euclidean conclusions.

In [ ]:
Gdep = nx.DiGraph()
edges = [
    ("circular points I,J", "Laguerre angle formula"),
    ("circular points I,J", "orthogonality at infinity"),
    ("circular points I,J", "mirror join/meet construction"),
    ("Laguerre angle formula", "angle-bisector bracket formula"),
    ("orthogonality at infinity", "altitudes concur"),
    ("mirror join/meet construction", "focal reflection test"),
    ("complex tangents from I,J", "foci of a conic"),
    ("focal reflection test", "foci of a conic"),
    ("degenerate Brianchon", "foci of a conic"),
    ("harmonic midpoint", "medians concur"),
    ("Menelaus/Ceva", "medians concur"),
    ("quadrilateral sets", "altitudes concur"),
    ("quadrilateral sets", "angle bisectors in triangle"),
    ("Desargues", "Euler line"),
    ("Pascal", "six midpoint conic"),
    ("Hesse transfer", "foot points on conic"),
    ("six midpoint conic", "hybrid nine-point conic"),
    ("foot points on conic", "hybrid nine-point conic"),
]
Gdep.add_edges_from(edges)
pos = {
    "circular points I,J": (0, 3.0),
    "Laguerre angle formula": (1.4, 3.8),
    "orthogonality at infinity": (1.4, 3.0),
    "mirror join/meet construction": (1.4, 2.2),
    "angle-bisector bracket formula": (3.2, 3.8),
    "altitudes concur": (3.2, 3.0),
    "focal reflection test": (3.2, 2.2),
    "complex tangents from I,J": (1.4, 1.45),
    "degenerate Brianchon": (3.2, 1.45),
    "foci of a conic": (4.9, 1.85),
    "harmonic midpoint": (0, 0.8),
    "Menelaus/Ceva": (1.4, 0.8),
    "medians concur": (3.2, 0.8),
    "quadrilateral sets": (1.4, -0.15),
    "angle bisectors in triangle": (3.2, -0.15),
    "Desargues": (0, -1.0),
    "Euler line": (1.4, -1.0),
    "Pascal": (0, -1.85),
    "Hesse transfer": (1.4, -1.85),
    "six midpoint conic": (3.2, -1.65),
    "foot points on conic": (3.2, -2.2),
    "hybrid nine-point conic": (5.0, -1.95),
}
fig, ax = plt.subplots(figsize=(12.2, 7.0))
node_colors = ["#fee2e2" if "circular" in node or node in {"complex tangents from I,J"} else "#dbeafe" if node in {"foci of a conic", "hybrid nine-point conic", "Euler line"} else "#f8fafc" for node in Gdep.nodes]
nx.draw_networkx_edges(Gdep, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=13, width=1.2, edge_color="#64748b")
nx.draw_networkx_nodes(Gdep, pos, ax=ax, node_color=node_colors, node_size=1850, edgecolors="#334155", linewidths=0.8)
nx.draw_networkx_labels(Gdep, pos, ax=ax, font_size=8.1)
ax.set_axis_off()
ax.set_title("Chapter 19 proof dependencies: where Euclidean facts enter projective arguments")
proof_artifact = save_fig(fig, "proof-dependency-map.png")
CHECKS["proof_dependency_graph"] = {"nodes": Gdep.number_of_nodes(), "edges": Gdep.number_of_edges(), "has_circular_points_node": "circular points I,J" in Gdep.nodes}
proof_artifact

In [ ]:
display_artifact(proof_artifact, width=920)

## Artifact Gallery

The links below make the chapter readable before execution and satisfy the course's direct-artifact contract.

![Mirror image via circular points](../../artifacts/chapter-19-euclidean-structures-from-a-projective-perspective/figures/mirror-image-circular-points.png)

![Angle bisectors via Laguerre formula](../../artifacts/chapter-19-euclidean-structures-from-a-projective-perspective/figures/angle-bisectors-laguerre-formula.png)

![Circle center from invisible tangents](../../artifacts/chapter-19-euclidean-structures-from-a-projective-perspective/figures/circle-center-invisible-tangents.png)

![Conic foci from circular tangents](../../artifacts/chapter-19-euclidean-structures-from-a-projective-perspective/figures/conic-foci-from-circular-tangents.png)

![Triangle theorem projective dashboard](../../artifacts/chapter-19-euclidean-structures-from-a-projective-perspective/figures/triangle-theorems-projective-dashboard.png)

![Hybrid nine-point conic lab](../../artifacts/chapter-19-euclidean-structures-from-a-projective-perspective/figures/hybrid-ninepoint-conic-lab.png)

Open the [confocal conic parameter lab](../../artifacts/chapter-19-euclidean-structures-from-a-projective-perspective/html/conics-through-point-from-foci.html) and the [hybrid residual table](../../artifacts/chapter-19-euclidean-structures-from-a-projective-perspective/tables/hybrid-ninepoint-residuals.csv).

## Applied Lab Checklist

1. In the mirror diagram, change the mirror line and confirm that the construction still agrees with ordinary reflection.
2. In the angle-bisector panel, move the two source directions closer together and watch the two bisectors remain perpendicular.
3. In the confocal HTML lab, move `P` and compare what stays fixed: the foci and the distance sum/difference rules.
4. In the hybrid nine-point lab, replace `H` by the orthocenter from the triangle panel. The conic should specialize to a circle, revealing the usual Euclidean theorem as a metric slice of a projective statement.

In [ ]:
sample = [-1.4, -0.2, 0.75, 1.6]
image = [(1.1 * x - 0.25) / (0.22 * x + 1.0) for x in sample]
cross_ratio_error = abs(cross_ratio(*sample) - cross_ratio(*image))
CHECKS["cross_ratio_error"] = float(cross_ratio_error)

manifest_rows = []
for artifact in ARTIFACTS:
    manifest_rows.append({
        "artifact": book_relative(artifact),
        "kind": artifact.suffix.lower().lstrip("."),
        "size": artifact.stat().st_size,
    })
manifest = save_table(manifest_rows, ARTIFACT_ROOT, "tables", "artifact-manifest.csv")
if manifest not in ARTIFACTS:
    ARTIFACTS.append(manifest)

raster_stats = [image_stats(path) for path in ARTIFACTS if path.suffix.lower() == ".png"]
visual_checks = {
    "all_files_exist": all(path.exists() and path.stat().st_size > 256 for path in ARTIFACTS),
    "chapter": 19,
    "cross_ratio_error": float(cross_ratio_error),
    "visual_count": len(ARTIFACTS),
    "raster_artifacts": [{**item, "path": book_relative(item["path"])} for item in raster_stats],
    "numeric_checks": CHECKS,
}
save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")
save_json(CHECKS, ARTIFACT_ROOT, "checks", "chapter19-invariants.json")
pd.DataFrame(manifest_rows)

## Takeaways

- The circular points are not visual decoration; they are the projective handle by which angle, orthogonality, reflection, circles, and foci become algebraic constructions.
- Some constructions have visible real outcomes even when the decisive objects are complex, as with `QI`, `QJ`, and the tangent lines used to recover foci.
- The triangle theorem section is a translation exercise: metric words become incidence, harmonic, bracket, or quadrilateral-set statements.
- Hybrid thinking is productive because it asks which parts of a Euclidean theorem are affine, projective, or genuinely metric. The nine-point circle becomes a conic before it becomes a circle.

In [ ]:
assert_artifacts(ARTIFACTS, min_size=256)
for stat in raster_stats:
    assert stat["width"] >= 200 and stat["height"] >= 150
    assert stat["pixel_std"] > 2.0

assert CHECKS["mirror"]["projective_vs_classical_reflection"] < 1e-9
assert CHECKS["mirror"]["midpoint_on_mirror_line"] < 1e-9
assert CHECKS["mirror"]["reflection_segment_perpendicular_to_mirror"] < 1e-9
assert CHECKS["mirror"]["symbolic_horizontal_reflection_ok"]
assert CHECKS["angle_bisectors"]["max_direction_mismatch"] < 1e-9
assert CHECKS["angle_bisectors"]["bisectors_perpendicular_dot"] < 1e-9
assert CHECKS["circle_center"]["center_residual"] < 1e-9
assert CHECKS["conic_foci"]["focus_coordinate_residual"] < 1e-9
assert CHECKS["conic_foci"]["max_reflection_law_residual"] < 1e-9
assert CHECKS["conics_from_foci"]["initial_point_ellipse_equation_residual"] < 1e-9
assert CHECKS["conics_from_foci"]["initial_point_hyperbola_equation_residual"] < 1e-9
assert CHECKS["triangle_theorems"]["median_concurrence_residual"] < 1e-9
assert CHECKS["triangle_theorems"]["altitude_concurrence_residual"] < 1e-9
assert CHECKS["triangle_theorems"]["angle_bisector_equal_angle_residual"] < 1e-9
assert CHECKS["triangle_theorems"]["euler_line_collinearity_residual"] < 1e-9
assert CHECKS["hybrid_ninepoint_conic"]["table_max_foot_residual"] < 1e-8
assert CHECKS["proof_dependency_graph"]["nodes"] >= 18
assert CHECKS["cross_ratio_error"] < 1e-9

final = {
    "chapter": 19,
    "notebook_executed": True,
    "source_span": CHECKS["source_span"],
    "artifacts": [book_relative(path) for path in ARTIFACTS],
    "checks": CHECKS,
}
save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
final